# Exploratory Data Analysis: Sierra Leone Solar Data

## Overview
This notebook performs comprehensive exploratory data analysis on the Sierra Leone solar farm dataset, including data profiling, cleaning, outlier detection, and visualization.

## Objectives
1. Load and inspect the dataset
2. Perform summary statistics and missing value analysis
3. Detect and handle outliers
4. Conduct time series analysis
5. Analyze cleaning impact
6. Explore correlations and relationships
7. Analyze wind patterns and distributions
8. Examine temperature relationships
9. Create visualizations
10. Export cleaned data


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import zscore
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")


## 1. Data Loading and Initial Inspection


In [ ]:
# Load the dataset
df = pd.read_csv('../data/sierra_leone.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nFirst few rows:")
df.head()


In [ ]:
# Convert Timestamp to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Display data types
print("Data Types:")
print(df.dtypes)
print("\nDataset Info:")
df.info()


## 2. Summary Statistics & Missing Value Report


In [ ]:
# Summary statistics for numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
print("Summary Statistics:")
print(df[numeric_cols].describe())


In [ ]:
# Missing values analysis
missing_values = df.isna().sum()
missing_percentage = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Count': missing_values.values,
    'Missing Percentage': missing_percentage.values
})

missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

print("Missing Values Report:")
print(missing_df)

# Identify columns with >5% missing values
high_missing = missing_df[missing_df['Missing Percentage'] > 5]
if len(high_missing) > 0:
    print("\nColumns with >5% missing values:")
    print(high_missing)
else:
    print("\nNo columns have >5% missing values.")


## 3. Outlier Detection & Basic Cleaning


In [ ]:
# Key columns for outlier detection
key_columns = ['GHI', 'DNI', 'DHI', 'ModA', 'ModB', 'WS', 'WSgust']

# Calculate Z-scores for key columns
z_scores = {}
outlier_flags = pd.DataFrame(index=df.index)

for col in key_columns:
    if col in df.columns:
        z_scores[col] = np.abs(zscore(df[col].dropna()))
        outlier_flags[col] = np.abs(zscore(df[col].fillna(df[col].median()))) > 3

# Count outliers per column
outlier_counts = outlier_flags.sum()
print("Outlier counts (|Z| > 3):")
print(outlier_counts)


In [ ]:
# Visualize outliers using boxplots
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for idx, col in enumerate(key_columns[:7]):
    if col in df.columns:
        df.boxplot(column=col, ax=axes[idx])
        axes[idx].set_title(f'Boxplot: {col}')
        axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

# Handle missing values: Impute with median for numeric columns
numeric_cols_to_impute = ['GHI', 'DNI', 'DHI', 'ModA', 'ModB', 'Tamb', 'RH', 'WS', 'WSgust', 'WD', 'BP', 'TModA', 'TModB']

for col in numeric_cols_to_impute:
    if col in df_clean.columns:
        if df_clean[col].isna().sum() > 0:
            median_value = df_clean[col].median()
            df_clean[col].fillna(median_value, inplace=True)
            print(f"Imputed {col} with median: {median_value:.2f}")

# Handle outliers: Cap extreme values (optional - can also remove)
for col in key_columns:
    if col in df_clean.columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Cap outliers instead of removing (preserves data)
        df_clean[col] = df_clean[col].clip(lower=lower_bound, upper=upper_bound)
        
print("\nOutlier handling completed.")


## 4. Time Series Analysis


In [ ]:
# Sort by timestamp
df_clean = df_clean.sort_values('Timestamp')

# Time series plots for key metrics
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# GHI over time
axes[0, 0].plot(df_clean['Timestamp'], df_clean['GHI'], alpha=0.7, linewidth=0.5)
axes[0, 0].set_title('GHI Over Time')
axes[0, 0].set_xlabel('Timestamp')
axes[0, 0].set_ylabel('GHI (W/m²)')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(True, alpha=0.3)

# DNI over time
axes[0, 1].plot(df_clean['Timestamp'], df_clean['DNI'], alpha=0.7, linewidth=0.5, color='orange')
axes[0, 1].set_title('DNI Over Time')
axes[0, 1].set_xlabel('Timestamp')
axes[0, 1].set_ylabel('DNI (W/m²)')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(True, alpha=0.3)

# DHI over time
axes[1, 0].plot(df_clean['Timestamp'], df_clean['DHI'], alpha=0.7, linewidth=0.5, color='green')
axes[1, 0].set_title('DHI Over Time')
axes[1, 0].set_xlabel('Timestamp')
axes[1, 0].set_ylabel('DHI (W/m²)')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, alpha=0.3)

# Ambient Temperature over time
axes[1, 1].plot(df_clean['Timestamp'], df_clean['Tamb'], alpha=0.7, linewidth=0.5, color='red')
axes[1, 1].set_title('Ambient Temperature Over Time')
axes[1, 1].set_xlabel('Timestamp')
axes[1, 1].set_ylabel('Tamb (°C)')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Monthly patterns
df_clean['Month'] = df_clean['Timestamp'].dt.month
df_clean['Hour'] = df_clean['Timestamp'].dt.hour

# Monthly average GHI
monthly_ghi = df_clean.groupby('Month')['GHI'].mean()

plt.figure(figsize=(12, 6))
monthly_ghi.plot(kind='bar', color='skyblue')
plt.title('Average GHI by Month')
plt.xlabel('Month')
plt.ylabel('Average GHI (W/m²)')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3)
plt.show()

# Hourly patterns
hourly_ghi = df_clean.groupby('Hour')['GHI'].mean()

plt.figure(figsize=(12, 6))
hourly_ghi.plot(kind='line', marker='o', color='orange')
plt.title('Average GHI by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Average GHI (W/m²)')
plt.grid(True, alpha=0.3)
plt.show()


## 5. Cleaning Impact Analysis


In [ ]:
# Analyze cleaning impact on ModA and ModB
if 'Cleaning' in df_clean.columns:
    cleaning_impact = df_clean.groupby('Cleaning')[['ModA', 'ModB']].mean()
    
    print("Average ModA and ModB by Cleaning Status:")
    print(cleaning_impact)
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    cleaning_impact['ModA'].plot(kind='bar', ax=axes[0], color='blue', alpha=0.7)
    axes[0].set_title('Average ModA: Pre vs Post Cleaning')
    axes[0].set_xlabel('Cleaning (0=No, 1=Yes)')
    axes[0].set_ylabel('Average ModA (W/m²)')
    axes[0].set_xticklabels(['No Cleaning', 'Cleaning'], rotation=0)
    axes[0].grid(True, alpha=0.3)
    
    cleaning_impact['ModB'].plot(kind='bar', ax=axes[1], color='green', alpha=0.7)
    axes[1].set_title('Average ModB: Pre vs Post Cleaning')
    axes[1].set_xlabel('Cleaning (0=No, 1=Yes)')
    axes[1].set_ylabel('Average ModB (W/m²)')
    axes[1].set_xticklabels(['No Cleaning', 'Cleaning'], rotation=0)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Cleaning column not found in dataset.")


## 6. Correlation & Relationship Analysis


In [ ]:
# Correlation heatmap
correlation_cols = ['GHI', 'DNI', 'DHI', 'TModA', 'TModB', 'Tamb', 'RH', 'WS', 'BP']
corr_data = df_clean[correlation_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_data, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=1)
plt.title('Correlation Heatmap: Key Variables')
plt.tight_layout()
plt.show()


In [ ]:
# Scatter plots: Wind vs GHI
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# WS vs GHI
axes[0].scatter(df_clean['WS'], df_clean['GHI'], alpha=0.5, s=10)
axes[0].set_xlabel('Wind Speed (m/s)')
axes[0].set_ylabel('GHI (W/m²)')
axes[0].set_title('Wind Speed vs GHI')
axes[0].grid(True, alpha=0.3)

# WSgust vs GHI
axes[1].scatter(df_clean['WSgust'], df_clean['GHI'], alpha=0.5, s=10, color='orange')
axes[1].set_xlabel('Wind Gust Speed (m/s)')
axes[1].set_ylabel('GHI (W/m²)')
axes[1].set_title('Wind Gust vs GHI')
axes[1].grid(True, alpha=0.3)

# WD vs GHI
axes[2].scatter(df_clean['WD'], df_clean['GHI'], alpha=0.5, s=10, color='green')
axes[2].set_xlabel('Wind Direction (°N)')
axes[2].set_ylabel('GHI (W/m²)')
axes[2].set_title('Wind Direction vs GHI')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# RH vs Tamb and RH vs GHI
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# RH vs Tamb
axes[0].scatter(df_clean['RH'], df_clean['Tamb'], alpha=0.5, s=10, color='purple')
axes[0].set_xlabel('Relative Humidity (%)')
axes[0].set_ylabel('Ambient Temperature (°C)')
axes[0].set_title('Relative Humidity vs Ambient Temperature')
axes[0].grid(True, alpha=0.3)

# RH vs GHI
axes[1].scatter(df_clean['RH'], df_clean['GHI'], alpha=0.5, s=10, color='red')
axes[1].set_xlabel('Relative Humidity (%)')
axes[1].set_ylabel('GHI (W/m²)')
axes[1].set_title('Relative Humidity vs GHI')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 7. Wind & Distribution Analysis


In [ ]:
# Wind rose (simplified using scatter plot)
# For a proper wind rose, install windrose package: pip install windrose

try:
    from windrose import WindroseAxes
    
    fig = plt.figure(figsize=(10, 8))
    ax = WindroseAxes.from_ax(fig=fig)
    ax.bar(df_clean['WD'], df_clean['WS'], normed=True, opening=0.8, edgecolor='white')
    ax.set_legend(title='Wind Speed (m/s)')
    ax.set_title('Wind Rose: Sierra Leone')
    plt.show()
except ImportError:
    # Fallback: Radial bar plot
    print("Windrose package not available. Creating simplified wind direction plot.")
    
    # Wind direction distribution
    wind_dir_bins = np.arange(0, 361, 22.5)
    wind_dir_counts, _ = np.histogram(df_clean['WD'].dropna(), bins=wind_dir_bins)
    
    angles = np.deg2rad(np.arange(0, 360, 22.5))
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
    ax.bar(angles, wind_dir_counts, width=np.deg2rad(22.5), alpha=0.7)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_title('Wind Direction Distribution', pad=20)
    plt.show()


In [ ]:
# Histograms for GHI and Wind Speed
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GHI histogram
axes[0].hist(df_clean['GHI'].dropna(), bins=50, color='skyblue', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('GHI (W/m²)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of GHI')
axes[0].grid(True, alpha=0.3)

# Wind Speed histogram
axes[1].hist(df_clean['WS'].dropna(), bins=50, color='orange', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Wind Speed (m/s)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Wind Speed')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Temperature Analysis


In [ ]:
# Temperature relationships
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# RH influence on Temperature
axes[0].scatter(df_clean['RH'], df_clean['Tamb'], alpha=0.5, s=10, c=df_clean['GHI'], cmap='viridis')
axes[0].set_xlabel('Relative Humidity (%)')
axes[0].set_ylabel('Ambient Temperature (°C)')
axes[0].set_title('RH vs Tamb (colored by GHI)')
axes[0].grid(True, alpha=0.3)
plt.colorbar(axes[0].collections[0], ax=axes[0], label='GHI (W/m²)')

# Module temperatures vs Ambient temperature
axes[1].scatter(df_clean['Tamb'], df_clean['TModA'], alpha=0.5, s=10, label='TModA', color='red')
axes[1].scatter(df_clean['Tamb'], df_clean['TModB'], alpha=0.5, s=10, label='TModB', color='blue')
axes[1].set_xlabel('Ambient Temperature (°C)')
axes[1].set_ylabel('Module Temperature (°C)')
axes[1].set_title('Module Temperatures vs Ambient Temperature')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 9. Bubble Chart


In [ ]:
# Sample data for bubble chart (to avoid overplotting)
df_sample = df_clean.sample(n=min(1000, len(df_clean)), random_state=42)

# Bubble chart: GHI vs Tamb with bubble size = RH
plt.figure(figsize=(12, 8))
scatter = plt.scatter(df_sample['Tamb'], df_sample['GHI'], 
                     s=df_sample['RH']*5,  # Bubble size proportional to RH
                     alpha=0.6, 
                     c=df_sample['BP'], 
                     cmap='viridis',
                     edgecolors='black', 
                     linewidth=0.5)
plt.xlabel('Ambient Temperature (°C)')
plt.ylabel('GHI (W/m²)')
plt.title('GHI vs Ambient Temperature\n(Bubble size = RH, Color = Barometric Pressure)')
plt.colorbar(scatter, label='Barometric Pressure (hPa)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 10. Data Export


In [ ]:
# Export cleaned dataframe
output_path = '../data/sierra_leone_clean.csv'
df_clean.to_csv(output_path, index=False)
print(f"Cleaned data exported to {output_path}")
print(f"Shape of cleaned data: {df_clean.shape}")
print(f"Missing values in cleaned data: {df_clean.isna().sum().sum()}")


## Summary

### Key Findings:
1. **Data Quality**: [Summary of missing values and data quality]
2. **Outliers**: [Summary of outlier detection and handling]
3. **Trends**: [Key temporal trends observed]
4. **Relationships**: [Key correlations and relationships]
5. **Solar Potential**: [Assessment of solar potential for Sierra Leone]

### Recommendations:
- [Actionable insights and recommendations]
